# Autocorrelation-Adjusted Forecasting

The lecture gives an approximate method for first-order autocorrelation:

1. Fit the ordinary regression model and record residuals $e_t = y_t - \hat y_t$.
2. Estimate $\phi$ from the regression $e_t = \phi e_{t-1} + a_t$ without an intercept.
3. For a forecast $\tau$ periods ahead, revise the ordinary fitted forecast by adding $\hat\phi^\tau e_T$.

This does not replace a full time-series model, but it connects the regression workflow to the independence problem.

Approximate prediction intervals use the innovation standard error $s$ and accumulated AR(1) uncertainty:

$$\hat y_{T+\tau,adj} \pm z_{1-\alpha/2}s\sqrt{1+\hat\phi^2+\cdots+\hat\phi^{2(\tau-1)}}.$$

For $\tau=1$, this reduces to $\hat y_{T+1,adj} \pm z_{1-\alpha/2}s$.

Here $z_{1-\alpha/2}$ denotes the upper standard-normal quantile used for a two-sided approximate prediction interval.

In [1]:
from lite_setup import ensure_packages
await ensure_packages()

Running outside JupyterLite; assuming packages are already installed.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from checks import check_columns, check_no_missing

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True
demand = pd.read_csv(Path('data/autocorrelated_demand.csv'))
base_model = smf.ols('Demand ~ Period + Load', data=demand).fit()
resid = base_model.resid.to_numpy()
T = int(demand['Period'].max())
last_resid = resid[-1]
print(base_model.summary().tables[1])

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     44.4252     11.421      3.890      0.000      21.555      67.296
Period         0.4358      0.130      3.349      0.001       0.175       0.696
Load          -0.0042      0.191     -0.022      0.983      -0.387       0.379


In [3]:
lagged = resid[:-1]
current = resid[1:]
phi_hat = float(np.sum(current * lagged) / np.sum(lagged ** 2))
innovation = current - phi_hat * lagged
s_innov = float(np.sqrt(np.sum(innovation ** 2) / (len(innovation) - 1)))
print(f'Estimated phi: {phi_hat:.3f}')
print(f'Innovation standard error: {s_innov:.3f}')
print(f'Last residual e_T: {last_resid:.3f}')

Estimated phi: 0.612
Innovation standard error: 1.606
Last residual e_T: -0.679


In [4]:
future = pd.DataFrame({
    'Period': np.arange(T + 1, T + 7),
    'Load': np.linspace(demand['Load'].iloc[-1] + 0.8, demand['Load'].iloc[-1] + 4.8, 6),
})
ordinary = base_model.get_prediction(future).summary_frame(alpha=0.05)
future['ordinary_forecast'] = ordinary['mean'].to_numpy()
future['tau'] = np.arange(1, len(future) + 1)
future['ar1_adjustment'] = (phi_hat ** future['tau']) * last_resid
future['adjusted_forecast'] = future['ordinary_forecast'] + future['ar1_adjustment']
future[['Period', 'Load', 'ordinary_forecast', 'ar1_adjustment', 'adjusted_forecast']].round(3)

,Period,Load,ordinary_forecast,ar1_adjustment,adjusted_forecast
0,61,102.436,70.583,-0.415,70.168
1,62,103.236,71.016,-0.254,70.762
2,63,104.036,71.448,-0.155,71.293
3,64,104.836,71.881,-0.095,71.786
4,65,105.636,72.313,-0.058,72.255
5,66,106.436,72.746,-0.036,72.710


In [5]:
z = stats.norm.ppf(0.975)
interval_rows = []
for tau, adjusted in zip(future['tau'], future['adjusted_forecast']):
    variance_multiplier = sum(phi_hat ** (2 * j) for j in range(tau))
    half_width = z * s_innov * np.sqrt(variance_multiplier)
    interval_rows.append({
        'tau': tau,
        'adjusted_forecast': adjusted,
        'approx_pi_lower': adjusted - half_width,
        'approx_pi_upper': adjusted + half_width,
    })
pd.DataFrame(interval_rows).round(3)

,tau,adjusted_forecast,approx_pi_lower,approx_pi_upper
0,1,70.168,67.020,73.316
1,2,70.762,67.072,74.452
2,3,71.293,67.419,75.166
3,4,71.786,67.846,75.726
4,5,72.255,68.290,76.220
5,6,72.710,68.736,76.684


Interpretation:

- If the last residual is positive and $\hat\phi$ is positive, near-term forecasts are adjusted upward.
- The adjustment fades as $\tau$ grows because $\hat\phi^\tau$ shrinks when $|\hat\phi| < 1$.
- Approximate intervals widen with the accumulated AR(1) uncertainty.

For serious forecasting work, this step should be followed by residual checks and, when appropriate, a dedicated time-series model. In this course module, the main point is to understand why independence matters and how first-order autocorrelation affects regression forecasts.